In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
df = pd.read_hdf("pems-bay.h5")

traffic = df.values

print(traffic.shape)

(52116, 325)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    - sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(52104, 12, 325)
(52104, 325)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
from torch.utils.data import (
    TensorDataset,
    DataLoader,
    random_split
)

full_train_dataset = TensorDataset(
    X_train,
    y_train
)

train_size = int(
    0.9 * len(full_train_dataset)
)

val_size = (
    len(full_train_dataset)
    - train_size
)

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64,
    shuffle=False
)

In [8]:
import torch
import torch.nn as nn

class TemporalConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(3,1),
            padding=(1,0)
        )

    def forward(self,x):

        return torch.relu(
            self.conv(x)
        )

In [9]:
class ClusterHypergraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        num_clusters,
        channels
    ):

        super().__init__()

        self.num_nodes = num_nodes
        self.num_clusters = num_clusters

        self.cluster_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                num_clusters
            )
        )

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self,x):

        H = torch.softmax(
            self.cluster_embeddings,
            dim=1
        )

        hyper_adj = H @ H.T

        hyper_adj = hyper_adj / (
            hyper_adj.sum(
                dim=1,
                keepdim=True
            )
            + 1e-6
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            hyper_adj,
            x
        )

        x = x.permute(
            0,
            2,
            3,
            1
        )

        x = self.weight(x)

        x = x.permute(
            0,
            3,
            1,
            2
        )

        return torch.relu(x)

In [10]:
class CAHSTGCN(nn.Module):

    def __init__(self):

        super().__init__()

        self.temp1 = TemporalConv(
            1,
            32
        )

        self.hypergraph = ClusterHypergraphConv(
            num_nodes=325,
            num_clusters=16,
            channels=32
        )

        self.temp2 = TemporalConv(
            32,
            32
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self,x):

        x = x.unsqueeze(1)

        x = self.temp1(x)

        x = self.hypergraph(x)

        x = self.temp2(x)

        x = x.mean(dim=2)

        x = x.permute(
            0,
            2,
            1
        )

        x = self.fc(x)

        return x.squeeze(-1)

In [11]:
model = CAHSTGCN()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 325])
torch.Size([64, 325])


In [12]:

model = CAHSTGCN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)

best_val_loss = float('inf')
patience = 15
counter = 0

import time
train_start = time.time()

epochs = 150

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(pred, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    total_loss /= len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:

            pred = model(X_batch)

            loss = criterion(pred, y_batch)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        counter = 0

        torch.save(
            model.state_dict(),
            "best_cah_stgcn.pth"
        )

    else:

        counter += 1

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Train: {total_loss:.6f} "
        f"Val: {val_loss:.6f} "
        f"LR: {optimizer.param_groups[0]['lr']:.6f}"
    )

    if counter >= patience:

        print(f"Early stopping at epoch {epoch+1}")
        break


Epoch 1/150 Train: 0.022308 Val: 0.010629 LR: 0.001000
Epoch 2/150 Train: 0.009211 Val: 0.007973 LR: 0.001000
Epoch 3/150 Train: 0.007326 Val: 0.006758 LR: 0.001000
Epoch 4/150 Train: 0.006251 Val: 0.005796 LR: 0.001000
Epoch 5/150 Train: 0.005623 Val: 0.005606 LR: 0.001000
Epoch 6/150 Train: 0.005229 Val: 0.005100 LR: 0.001000
Epoch 7/150 Train: 0.004954 Val: 0.004918 LR: 0.001000
Epoch 8/150 Train: 0.004794 Val: 0.004897 LR: 0.001000
Epoch 9/150 Train: 0.004674 Val: 0.004612 LR: 0.001000
Epoch 10/150 Train: 0.004596 Val: 0.004567 LR: 0.001000
Epoch 11/150 Train: 0.004538 Val: 0.004454 LR: 0.001000
Epoch 12/150 Train: 0.004457 Val: 0.004485 LR: 0.001000
Epoch 13/150 Train: 0.004396 Val: 0.004349 LR: 0.001000
Epoch 14/150 Train: 0.004348 Val: 0.004300 LR: 0.001000
Epoch 15/150 Train: 0.004308 Val: 0.004298 LR: 0.001000
Epoch 16/150 Train: 0.004284 Val: 0.004268 LR: 0.001000
Epoch 17/150 Train: 0.004259 Val: 0.004215 LR: 0.001000
Epoch 18/150 Train: 0.004248 Val: 0.004323 LR: 0.001000
E

In [13]:
train_time = time.time() - train_start
print("Time Taken:", train_time)

Time Taken: 19788.302631616592


In [14]:
model.load_state_dict(torch.load("best_cah_stgcn.pth"))

<All keys matched successfully>

In [15]:
torch.save(
    model.state_dict(),
    "CAH-STGCN-METRLa.pth"
)

In [16]:
test_dataset = TensorDataset(
    X_test,
    y_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_targets = []

model.eval()

infer_start = time.time()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        pred = model(X_batch)

        all_predictions.append(
            pred.numpy()
        )

        all_targets.append(
            y_batch.numpy()
        )

predictions = np.concatenate(
    all_predictions,
    axis=0
)

infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = np.concatenate(
    all_targets,
    axis=0
)

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

Infer Time: 11.837214469909668
MAE: 0.037252455949783325
RMSE: 0.06565322694668392


In [17]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)

print("MAPE:", mape)
print("R2:", r2)

MAPE: 6628.802
R2: 0.7702093720436096


In [18]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 9521
